# Colab 모델 서버
표정 분석 (MediaPipe blendshape + RandomForest) + 핸드폰 감지 (YOLO)를  
FastAPI로 서빙하고 ngrok으로 외부에 노출

## 1. 패키지 설치

In [1]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio
!pip install -q mediapipe==0.10.21 opencv-python-headless
!pip install -q ultralytics scikit-learn joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 11.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompa

## 2. Google Drive 마운트 & 모델 로드

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# ── Drive 경로 설정 ───────────────────────────────────────────────────────────
DRIVE_MODEL_DIR = '/content/drive/MyDrive/model_pt'

EXPRESSION_RF_PATH   = os.path.join(DRIVE_MODEL_DIR, 'blendshape_rf_emotion.joblib')
EXPRESSION_META_PATH = os.path.join(DRIVE_MODEL_DIR, 'blendshape_meta.json')
FACE_LANDMARKER_PATH = os.path.join(DRIVE_MODEL_DIR, 'face_landmarker.task')
PHONE_MODEL_PATH     = os.path.join(DRIVE_MODEL_DIR, 'phone_detection.pt')  

for p in [EXPRESSION_RF_PATH, EXPRESSION_META_PATH, FACE_LANDMARKER_PATH, PHONE_MODEL_PATH]:
    status = '✅' if os.path.exists(p) else '❌ 없음'
    print(f'{status}  {p}')

✅  /content/drive/MyDrive/model_pt/blendshape_rf_emotion.joblib
✅  /content/drive/MyDrive/model_pt/blendshape_meta.json
✅  /content/drive/MyDrive/model_pt/face_landmarker.task
✅  /content/drive/MyDrive/model_pt/best.pt


In [3]:
!pip install "numpy<2"

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.3
    Uninstalling numpy-2.4.3:
      Successfully uninstalled numpy-2.4.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2

In [4]:
!pip install --upgrade mediapipe "numpy<2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
  Attempting uninstall: mediapipe
    Found existing installation: mediapipe 0.10.21
    Uninstalling mediapipe-0.10.21:
      Successfully uninstalled mediapipe-0.10.21
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.


In [ ]:
import cv2
import json
import numpy as np
import joblib
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from ultralytics import YOLO

# ── 핸드폰 감지 confidence 임계값 ─────────────────────────────────────────────
PHONE_CONF_THRESH = 0.70   # 이 값 이상일 때만 핸드폰으로 판정

# ── 표정 모델: blendshape 이름 & 클래스 ──────────────────────────────────────
BLENDSHAPE_NAMES = [
    '_neutral','browDownLeft','browDownRight','browInnerUp','browOuterUpLeft','browOuterUpRight',
    'cheekPuff','cheekSquintLeft','cheekSquintRight','eyeBlinkLeft','eyeBlinkRight','eyeLookDownLeft',
    'eyeLookDownRight','eyeLookInLeft','eyeLookInRight','eyeLookOutLeft','eyeLookOutRight','eyeLookUpLeft',
    'eyeLookUpRight','eyeSquintLeft','eyeSquintRight','eyeWideLeft','eyeWideRight','jawForward','jawLeft',
    'jawOpen','jawRight','mouthClose','mouthDimpleLeft','mouthDimpleRight','mouthFrownLeft','mouthFrownRight',
    'mouthFunnel','mouthLeft','mouthLowerDownLeft','mouthLowerDownRight','mouthPressLeft','mouthPressRight',
    'mouthPucker','mouthRight','mouthRollLower','mouthRollUpper','mouthShrugLower','mouthShrugUpper',
    'mouthSmileLeft','mouthSmileRight','mouthStretchLeft','mouthStretchRight','mouthUpperUpLeft',
    'mouthUpperUpRight','noseSneerLeft','noseSneerRight'
]

EMOTION_CLASSES = ['engagement', 'boredom', 'confusion', 'amused', 'surprise', 'neutral']
EMOTION_KR = {
    'engagement': '집중',
    'boredom':    '지루함',
    'confusion':  '혼란',
    'amused':     '웃음',
    'surprise':   '놀람',
    'neutral':    '중립',
}

# ── MediaPipe FaceLandmarker 생성 ────────────────────────────────────────────
BaseOptions          = python.BaseOptions
FaceLandmarker       = vision.FaceLandmarker
FaceLandmarkerOptions = vision.FaceLandmarkerOptions
VisionRunningMode    = vision.RunningMode

face_landmarker = FaceLandmarker.create_from_options(
    FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=FACE_LANDMARKER_PATH),
        running_mode=VisionRunningMode.IMAGE,
        num_faces=1,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=False,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
)
print('✅ FaceLandmarker 로드 완료')

# ── RandomForest 표정 모델 로드 ───────────────────────────────────────────────
rf_model = joblib.load(EXPRESSION_RF_PATH)
print('✅ RandomForest 표정 모델 로드 완료')

# ── YOLO 핸드폰 감지 모델 로드 (YOLOv11s fine-tuned) ────────────────────────
yolo_model = YOLO(PHONE_MODEL_PATH)
print('✅ YOLO 핸드폰 감지 모델 로드 완료')

## 3. 추론 함수 정의

In [ ]:
# ── blendshape 추출 유틸 ──────────────────────────────────────────────────────
def extract_blendshapes(frame_bgr):
    """BGR 프레임 → blendshape dict (얼굴 미감지 시 None)"""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = face_landmarker.detect(mp_img)
    if not result.face_blendshapes:
        return None
    row = {name: 0.0 for name in BLENDSHAPE_NAMES}
    for item in result.face_blendshapes[0]:
        row[item.category_name] = float(item.score)
    return row


# ── 규칙 보정 함수 ────────────────────────────────────────────────────────────
def normalize_probs(p):
    p = np.maximum(np.asarray(p, dtype=np.float64), 1e-9)
    return p / p.sum()


def apply_rule_boost(probs, blend):
    probs = np.asarray(probs, dtype=np.float64).copy()
    idx = {e: i for i, e in enumerate(EMOTION_CLASSES)}

    smile      = (blend.get('mouthSmileLeft', 0) + blend.get('mouthSmileRight', 0)) / 2
    jaw_open   = blend.get('jawOpen', 0)
    brow_up    = blend.get('browInnerUp', 0)
    eye_wide   = (blend.get('eyeWideLeft', 0) + blend.get('eyeWideRight', 0)) / 2
    eye_blink  = (blend.get('eyeBlinkLeft', 0) + blend.get('eyeBlinkRight', 0)) / 2
    frown      = (blend.get('mouthFrownLeft', 0) + blend.get('mouthFrownRight', 0)) / 2
    eye_squint = (blend.get('eyeSquintLeft', 0) + blend.get('eyeSquintRight', 0)) / 2

    if smile > 0.45:
        probs[idx['amused']]  += 0.18 + 0.25 * smile
        probs[idx['boredom']] *= 0.90
    if jaw_open > 0.35 and (brow_up > 0.20 or eye_wide > 0.25):
        probs[idx['surprise']] += 0.16 + 0.20 * jaw_open + 0.15 * brow_up
    if eye_blink > 0.55 and smile < 0.20 and jaw_open < 0.15:
        probs[idx['boredom']] += 0.10 + 0.10 * eye_blink
    if frown > 0.20 and brow_up > 0.15:
        probs[idx['confusion']] += 0.10 + 0.15 * frown
    if smile < 0.25 and eye_blink < 0.35 and eye_squint < 0.35 and jaw_open < 0.20:
        probs[idx['engagement']] += 0.10
    if smile < 0.15 and jaw_open < 0.12 and brow_up < 0.12 and frown < 0.12:
        probs[idx['neutral']] += 0.12

    return normalize_probs(probs)


# ── 표정 추론 ─────────────────────────────────────────────────────────────────
def predict_expression(frame_bgr):
    blend = extract_blendshapes(frame_bgr)
    if blend is None:
        return {
            'face_found': False,
            'emotion': None,
            'emotion_kr': None,
            'confidence': 0.0,
            'emotion_probs': {e: 0.0 for e in EMOTION_CLASSES},
        }

    x     = np.array([blend.get(n, 0.0) for n in BLENDSHAPE_NAMES], dtype=np.float32).reshape(1, -1)
    probs = rf_model.predict_proba(x)[0]
    probs = apply_rule_boost(probs, blend)

    pred_idx = int(np.argmax(probs))
    emotion  = EMOTION_CLASSES[pred_idx]

    return {
        'face_found':    True,
        'emotion':       emotion,
        'emotion_kr':    EMOTION_KR[emotion],
        'confidence':    round(float(probs[pred_idx]), 4),
        'emotion_probs': {EMOTION_CLASSES[i]: round(float(probs[i]), 4) for i in range(len(EMOTION_CLASSES))},
    }


# ── 핸드폰 감지: 형태 기반 후처리 필터 ────────────────────────────────────────
# 폰의 물리적 특성:
#   - 종횡비(W/H): portrait 0.35~0.65 / landscape 1.5~2.8  → 정사각형(0.7~1.4)이면 폰 아님
#   - 최소 면적: 프레임 전체의 0.5% 이상 (너무 작으면 노이즈)
#   - 최대 면적: 프레임 전체의 60% 이하 (화면 전체를 덮으면 오탐)
PHONE_CONF_THRESH  = 0.40   # 낮게 잡고 후처리로 걸러냄
PHONE_AR_PORTRAIT  = (0.30, 0.68)   # portrait W/H 범위
PHONE_AR_LANDSCAPE = (1.45, 2.90)   # landscape W/H 범위
PHONE_AREA_MIN     = 0.005  # 프레임 면적의 0.5%
PHONE_AREA_MAX     = 0.60   # 프레임 면적의 60%

# 시간적 투표: 최근 3프레임 중 2회 이상 감지 시 확정
_phone_vote_window = []
PHONE_VOTE_N       = 3   # 윈도우 크기
PHONE_VOTE_K       = 2   # 확정에 필요한 최소 감지 횟수


def _is_phone_shape(box_xywh, frame_h, frame_w):
    """YOLO xyxy box가 폰의 형태 조건을 만족하는지 확인"""
    x1, y1, x2, y2 = box_xywh
    w = x2 - x1
    h = y2 - y1
    if h == 0:
        return False
    ar = w / h
    area_ratio = (w * h) / (frame_w * frame_h)

    # 크기 범위 체크
    if not (PHONE_AREA_MIN <= area_ratio <= PHONE_AREA_MAX):
        return False

    # 종횡비 체크: portrait 또는 landscape 범위에 속해야 함
    in_portrait   = PHONE_AR_PORTRAIT[0]   <= ar <= PHONE_AR_PORTRAIT[1]
    in_landscape  = PHONE_AR_LANDSCAPE[0]  <= ar <= PHONE_AR_LANDSCAPE[1]
    return in_portrait or in_landscape


def predict_phone(frame_bgr):
    """
    낮은 conf로 넓게 감지 → 형태 필터로 오탐 제거 → 시간적 투표로 확정.

    return: {'phone_detected': bool, 'phone_confidence': float}
    """
    global _phone_vote_window
    h, w = frame_bgr.shape[:2]

    results = yolo_model(frame_bgr, conf=PHONE_CONF_THRESH, verbose=False)
    boxes   = results[0].boxes

    best_conf   = 0.0
    this_detect = False

    if boxes is not None and len(boxes) > 0:
        for box in boxes:
            conf = float(box.conf[0])
            xyxy = box.xyxy[0].tolist()
            if _is_phone_shape(xyxy, h, w):
                if conf > best_conf:
                    best_conf = conf
                this_detect = True

    # 시간적 투표 업데이트
    _phone_vote_window.append(this_detect)
    if len(_phone_vote_window) > PHONE_VOTE_N:
        _phone_vote_window.pop(0)

    voted = sum(_phone_vote_window) >= PHONE_VOTE_K

    return {
        'phone_detected':   voted,
        'phone_confidence': round(best_conf, 4),
    }

print('✅ 추론 함수 정의 완료')

## 4. FastAPI 서버 정의

In [4]:
import time
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import JSONResponse

app = FastAPI(title='Focus Analyzer - Colab Model Server')


@app.get('/health')
def health():
    """연결 확인용. detection/main.py의 ColabSender가 내부적으로 사용."""
    import torch
    return {'status': 'ok', 'device': 'cuda' if torch.cuda.is_available() else 'cpu'}


@app.post('/analyze')
async def analyze(file: UploadFile = File(...)):
    """
    JPEG 프레임을 받아 표정 분석 + 핸드폰 감지 결과를 반환한다.

    Request : multipart/form-data  field='file'  (JPEG bytes)
    Response JSON:
    {
        "face_found"       : bool,
        "emotion"          : str | null,    # engagement / boredom / confusion / amused / surprise / neutral
        "emotion_kr"       : str | null,    # 집중 / 지루함 / 혼란 / 웃음 / 놀람 / 중립
        "confidence"       : float,
        "emotion_probs"    : dict,
        "phone_detected"   : bool,
        "phone_confidence" : float,
        "inference_ms"     : float
    }
    """
    if file.content_type not in ('image/jpeg', 'image/png'):
        raise HTTPException(status_code=400, detail='JPEG 또는 PNG만 허용됩니다.')

    img_bytes = await file.read()
    img_array = np.frombuffer(img_bytes, np.uint8)
    frame_bgr = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

    if frame_bgr is None:
        raise HTTPException(status_code=400, detail='이미지 디코딩 실패')

    t0 = time.perf_counter()
    expr_result  = predict_expression(frame_bgr)
    phone_result = predict_phone(frame_bgr)
    inference_ms = round((time.perf_counter() - t0) * 1000, 1)

    return JSONResponse({
        **expr_result,
        **phone_result,
        'inference_ms': inference_ms,
    })

print('✅ FastAPI 앱 정의 완료')

✅ FastAPI 앱 정의 완료


## 5. ngrok 설정 & 서버 시작

In [5]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok
from google.colab import userdata

# ── ngrok authtoken ───────────────────────────────────────────────────────────
NGROK_TOKEN = userdata.get('ngrok_auth_token')
ngrok.set_auth_token(NGROK_TOKEN)

# ── 터널 열기 ─────────────────────────────────────────────────────────────────
PORT = 8000
public_url = ngrok.connect(PORT, bind_tls=True)

print('=' * 60)
print(f'  Colab 서버 URL : {public_url}')
print('=' * 60)
print('  detection/main.py 에서 아래처럼 설정하세요:')
print(f'    COLAB_ENABLED = True')
print(f'    COLAB_URL     = "{public_url}"')
print('=' * 60)
print(f'  헬스체크: {public_url}/health')
print(f'  API 문서: {public_url}/docs')
print('=' * 60)

# ── 서버 실행 (블로킹) ────────────────────────────────────────────────────────
nest_asyncio.apply()
config = uvicorn.Config(app, host='0.0.0.0', port=PORT, log_level='warning')
server = uvicorn.Server(config)
await server.serve()

  Colab 서버 URL : NgrokTunnel: "https://waylon-unfancy-overidly.ngrok-free.dev" -> "http://localhost:8000"
  detection/main.py 에서 아래처럼 설정하세요:
    COLAB_ENABLED = True
    COLAB_URL     = "NgrokTunnel: "https://waylon-unfancy-overidly.ngrok-free.dev" -> "http://localhost:8000""
  헬스체크: NgrokTunnel: "https://waylon-unfancy-overidly.ngrok-free.dev" -> "http://localhost:8000"/health
  API 문서: NgrokTunnel: "https://waylon-unfancy-overidly.ngrok-free.dev" -> "http://localhost:8000"/docs


In [ ]:
import cv2
from ultralytics import YOLO

# 로컬 모델 경로 (수정 필요)
model = YOLO('phone_detection.pt')

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    results = model.track(
        frame,
        persist=True,
        conf=0.70,          # colab_server와 동일한 임계값
        iou=0.5,
        imgsz=320,
        tracker="botsort.yaml",
        verbose=False,
    )

    for r in results:
        annotated_frame = r.plot()

        if r.boxes.id is not None:
            ids = r.boxes.id.int().cpu().tolist()
            cv2.putText(annotated_frame, f"Tracking: {len(ids)}",
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("YOLOv11s Phone Tracking", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()